# 機械学習の前提として学ぶ NumPy / pandas 入門

このノートブックは、機械学習に入る前に必要になる `numpy` と `pandas` の基本を 1 時間で確認するための資料です。

狙いは、モデルを作る前に次の流れを理解することです。

1. 数値データを配列で持つ
2. 表データを DataFrame で扱う
3. 特徴量 `X` と目的変数 `y` を分ける
4. 簡単な前処理をする


## この講義のゴール

- `numpy.ndarray` が機械学習でよく使われる理由を理解する
- `pandas.DataFrame` から必要な列を選べる
- 特徴量 `X` と目的変数 `y` を作れる
- 欠損値、カテゴリ列、スケーリングの初歩を理解する


## 準備

もし `numpy` や `pandas` が未導入なら、次のセルを先に実行してください。

In [ ]:
# 必要なら先に実行
# %pip install numpy pandas


In [ ]:
import numpy as np
import pandas as pd

print('NumPy version:', np.__version__)
print('pandas version:', pd.__version__)

## 1. 機械学習でのデータの見方

機械学習では、データを大きく 2 つに分けて考えます。

- 特徴量 `X`: モデルへの入力
- 目的変数 `y`: 予測したい答え

例えば住宅価格予測なら、部屋数や面積が `X`、価格が `y` です。

In [ ]:
df = pd.DataFrame({
    'study_hours': [1, 2, 3, 4, 5, 6, 7, 8],
    'attendance': [60, 62, 70, 72, 80, 85, 90, 96],
    'mock_test': [45, 50, 52, 60, 65, 72, 78, 88],
    'club': ['N', 'Y', 'N', 'N', 'Y', 'N', 'Y', 'N'],
    'passed': [0, 0, 0, 0, 1, 1, 1, 1]
})

df

## 2. NumPy の基本: ベクトルと行列

機械学習では、特徴量を数値の行列として扱うことが多くあります。

- 1 人分のデータ: ベクトル
- 全員分のデータ: 行列


In [ ]:
student_a = np.array([4, 72, 60])
X_small = np.array([
    [1, 60, 45],
    [2, 62, 50],
    [3, 70, 52],
    [4, 72, 60]
])

print('student_a:', student_a)
print('student_a shape:', student_a.shape)
print('\nX_small:\n', X_small)
print('X_small shape:', X_small.shape)

### 配列全体にまとめて計算する

NumPy では、各要素に同じ計算を一度に適用できます。前処理で頻出です。

In [ ]:
hours = np.array([1, 2, 3, 4, 5])

print('hours:', hours)
print('hours * 2:', hours * 2)
print('hours + 1:', hours + 1)
print('平均:', hours.mean())

### スケーリングの感覚をつかむ

特徴量どうしのスケールが大きく違うと、モデル学習に影響することがあります。

ここでは平均 0、標準偏差 1 に近づける標準化を NumPy で体験します。

In [ ]:
mock = np.array([45, 50, 52, 60, 65, 72, 78, 88])
mock_std = (mock - mock.mean()) / mock.std()

print('元データ:', mock)
print('平均:', round(mock.mean(), 2))
print('標準偏差:', round(mock.std(), 2))
print('標準化後:', np.round(mock_std, 2))

## 3. pandas で表データを扱う

CSV や表形式データの読み込み後、最初に確認するのは次のような情報です。

- 行数と列数
- 列名
- 数値列と文字列列
- 欠損値の有無


In [ ]:
print(df.head())
print('\nshape:', df.shape)
print('columns:', df.columns.tolist())

In [ ]:
df.dtypes

## 4. 特徴量 `X` と目的変数 `y` を分ける

分類でも回帰でも、まずは入力と答えを分けるところから始めます。

In [ ]:
X = df[['study_hours', 'attendance', 'mock_test', 'club']]
y = df['passed']

print('X:')
print(X)
print('\ny:')
print(y)

### 数値列だけを取り出す

そのままでは文字列列を含むので、まず数値列だけを見る場面が多いです。

In [ ]:
X_num = df[['study_hours', 'attendance', 'mock_test']]
X_num

### NumPy 配列へ変換する

多くの機械学習ライブラリは、内部的には NumPy 配列に近い形でデータを扱います。

In [ ]:
X_array = X_num.to_numpy()
y_array = y.to_numpy()

print('X_array:\n', X_array)
print('X_array shape:', X_array.shape)
print('\ny_array:', y_array)
print('y_array shape:', y_array.shape)

## 5. 前処理 1: 欠損値を扱う

実データでは空欄がよくあります。まずは欠損値を見つけることが重要です。

In [ ]:
df_missing = df.copy()
df_missing.loc[2, 'mock_test'] = np.nan
df_missing.loc[5, 'attendance'] = np.nan

df_missing

In [ ]:
print(df_missing.isnull().sum())

In [ ]:
df_filled = df_missing.copy()
df_filled['attendance'] = df_filled['attendance'].fillna(df_filled['attendance'].mean())
df_filled['mock_test'] = df_filled['mock_test'].fillna(df_filled['mock_test'].mean())

df_filled

## 6. 前処理 2: カテゴリ列を数値化する

多くのモデルは文字列をそのまま扱えません。そこでカテゴリ列を数値に変換します。

In [ ]:
df_encoded = df_filled.copy()
df_encoded['club'] = df_encoded['club'].map({'N': 0, 'Y': 1})

df_encoded

## 7. 学習に使う形へまとめる

ここで、最終的な特徴量行列 `X_final` と目的変数 `y_final` を作ります。

In [ ]:
feature_cols = ['study_hours', 'attendance', 'mock_test', 'club']
X_final = df_encoded[feature_cols]
y_final = df_encoded['passed']

print('X_final:')
print(X_final)
print('\ny_final:')
print(y_final.values)

In [ ]:
X_final_np = X_final.to_numpy()
y_final_np = y_final.to_numpy()

print('X_final_np shape:', X_final_np.shape)
print('y_final_np shape:', y_final_np.shape)
print('\n1件目の特徴量:', X_final_np[0])
print('1件目の正解ラベル:', y_final_np[0])

## 8. 学習前の簡単な確認

学習前には、クラスの偏りや列の傾向をざっと見ることが重要です。

In [ ]:
print('passed の件数:')
print(df_encoded['passed'].value_counts())

In [ ]:
df_encoded.groupby('passed')[['study_hours', 'attendance', 'mock_test']].mean()

## ミニ演習

1. `study_hours` が 5 以上の行だけ取り出す
2. `attendance` を 100 で割って 0〜1 の値にする列を追加する
3. `club` をダミー変数化する別の方法として `pd.get_dummies()` を試す


In [ ]:
# ここに演習コードを書いてください


## まとめ

機械学習の前処理として重要なのは次の流れです。

- 表データを `DataFrame` として読む
- 特徴量 `X` と目的変数 `y` を分ける
- 欠損値やカテゴリ列を処理する
- 必要に応じて NumPy 配列へ変換する

次のステップとしては、`scikit-learn` で `train_test_split` やモデル学習につなげると自然です。